# 05 — Local RAG: FAISS + qwen3-embedding + Ollama LLM

A fully offline Retrieval-Augmented Generation pipeline — zero cloud calls, no LangChain,
no external services. Every component runs locally:

| Stage | Tool | Model / Library |
| --- | --- | --- |
| Chunking | `utils.helpers.chunk_text()` | word-based, 512 words, 64 overlap |
| Embedding | `ollama.embed()` | `qwen3-embedding:0.6b` (MTEB #1 multilingual) |
| Indexing | `faiss.IndexFlatL2` | exact L2 search on normalised vectors ≈ cosine |
| Retrieval | FAISS `index.search()` | top-k = 5 |
| Generation | `ollama.chat()` | `mistral:7b` with retrieved context |

Source document: `data/contoso_report_q4_2024.md` — the confidential Contoso HR report.

> **Why FAISS over ChromaDB?** Lab 05 already uses ChromaDB. FAISS is the lower-level
> alternative — no daemon, no HTTP server, pure Python. Using it here shows understanding
> of the vector search layer beneath framework abstractions.

In [1]:
from pathlib import Path

from utils.helpers import check_model_available, check_ollama_running

EMBED_MODEL = "qwen3-embedding:0.6b"
LLM_MODEL = "mistral:7b"
CHUNK_SIZE = 512      # words per chunk
CHUNK_OVERLAP = 64    # words of overlap between consecutive chunks
TOP_K = 5             # chunks retrieved per query

if not check_ollama_running():
    raise RuntimeError("Ollama is not running. Start it with: ollama serve")

for model in (EMBED_MODEL, LLM_MODEL):
    if not check_model_available(model):
        raise RuntimeError(
            f"Model '{model}' not found locally.\n"
            f"Pull it with: ollama pull {model}"
        )

print(f"✓ Embedding model: {EMBED_MODEL}")
print(f"✓ LLM model:       {LLM_MODEL}")

REPORT = Path("data/contoso_report_q4_2024.md").read_text()
print(f"✓ Document loaded: {len(REPORT):,} chars")

✓ Embedding model: qwen3-embedding:0.6b
✓ LLM model:       mistral:7b
✓ Document loaded: 5,585 chars


## 1. Chunking

Split the document into overlapping word-based chunks using `chunk_text()` from
`utils/helpers.py`. Overlap ensures that sentences spanning chunk boundaries are
still retrievable.

In [2]:
from utils.helpers import chunk_text

chunks = chunk_text(REPORT, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

print(f"Document → {len(chunks)} chunks")
print(f"Chunk size: {CHUNK_SIZE} words, overlap: {CHUNK_OVERLAP} words\n")
print("Sample — chunk 0 (first 200 chars):")
print(f"  {chunks[0][:200]}…")
print("\nSample — chunk 1 (first 200 chars):")
print(f"  {chunks[1][:200]}…")

Document → 3 chunks
Chunk size: 512 words, overlap: 64 words

Sample — chunk 0 (first 200 chars):
  # Contoso Corp — Internal HR & Financial Report Q4 2024 **Classification: CONFIDENTIAL — Internal Use Only** *This document contains sensitive employee and compensation data. Do not share externally.*…

Sample — chunk 1 (first 200 chars):
  HR | £55,900 | --- ## 5. Attrition — Q4 2024 **Q4 2024 attrition rate**: 4.2% (9 departures out of 214 average headcount) **Annualised attrition rate (2024)**: 14.8% **Industry benchmark (UK SaaS)**: …


## 2. Embedding

`ollama.embed()` accepts a list of strings and returns a list of embedding vectors.
We embed all chunks in a single batch call, then read the embedding dimension from
the first result to configure the FAISS index.

In [3]:
import time

import numpy as np
import ollama

print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL}…")
t0 = time.time()

embed_response = ollama.embed(model=EMBED_MODEL, input=chunks)
raw_embeddings = embed_response.embeddings   # list[list[float]]

elapsed = time.time() - t0
dimension = len(raw_embeddings[0])

print(f"Done in {elapsed:.1f}s")
print(f"Vectors: {len(raw_embeddings)} × {dimension} dimensions")
print(f"Speed:   {len(chunks) / elapsed:.1f} chunks/sec")

Embedding 3 chunks with qwen3-embedding:0.6b…
Done in 1.7s
Vectors: 3 × 1024 dimensions
Speed:   1.7 chunks/sec


## 3. FAISS index

`IndexFlatL2` performs exact brute-force L2 search. After normalising vectors with
`faiss.normalize_L2()`, L2 distance and cosine similarity rank results identically —
smaller L2 distance = higher cosine similarity.

In [4]:
import faiss

# Build float32 matrix — FAISS requires float32
vectors = np.array(raw_embeddings, dtype=np.float32)

# Normalise in-place so L2 distance ≈ cosine similarity ranking
faiss.normalize_L2(vectors)

index = faiss.IndexFlatL2(dimension)

if index.ntotal > 0:
    raise RuntimeError("Index already populated — re-run from the embedding cell.")

index.add(vectors)

print("FAISS index built")
print("  Type:      IndexFlatL2")
print(f"  Dimension: {dimension}")
print(f"  Vectors:   {index.ntotal}")

FAISS index built
  Type:      IndexFlatL2
  Dimension: 1024
  Vectors:   3


## 4. Retrieval

Embed the query, normalise it, then call `index.search()`. FAISS returns distances
and chunk indices sorted from most to least similar.

In [7]:
def retrieve(query: str, k: int = TOP_K) -> list[tuple[str, float]]:
    """Return the top-k most relevant chunks for *query* with their L2 distances."""
    if index.ntotal == 0:
        raise RuntimeError("Index is empty — run the indexing cell first.")

    q_resp = ollama.embed(model=EMBED_MODEL, input=[query])
    q_vec = np.array(q_resp.embeddings, dtype=np.float32)
    faiss.normalize_L2(q_vec)

    distances, indices = index.search(q_vec, k)
    pairs = zip(indices[0], distances[0], strict=True)
    return [(chunks[i], float(d)) for i, d in pairs]


# Quick retrieval demo
sample_query = "What is the Q4 attrition rate?"
results = retrieve(sample_query)

print(f"Query: '{sample_query}'\n")
for rank, (chunk, dist) in enumerate(results, 1):
    preview = chunk[:120].replace("\n", " ")
    print(f"  [{rank}] dist={dist:.4f}  {preview}…")

Query: 'What is the Q4 attrition rate?'

  [1] dist=0.5808  HR | £55,900 | --- ## 5. Attrition — Q4 2024 **Q4 2024 attrition rate**: 4.2% (9 departures out of 214 average headcount…
  [2] dist=0.8660  # Contoso Corp — Internal HR & Financial Report Q4 2024 **Classification: CONFIDENTIAL — Internal Use Only** *This docum…
  [3] dist=0.9788  | | Sales | Account Executive EMEA (×4), SDR (×3) | 7 | | Customer Success | Customer Success Manager (×3) | 3 | | Produ…
  [4] dist=340282346638528859811704183484516925440.0000  | | Sales | Account Executive EMEA (×4), SDR (×3) | 7 | | Customer Success | Customer Success Manager (×3) | 3 | | Produ…
  [5] dist=340282346638528859811704183484516925440.0000  | | Sales | Account Executive EMEA (×4), SDR (×3) | 7 | | Customer Success | Customer Success Manager (×3) | 3 | | Produ…


## 5. Generation

Inject the retrieved chunks into the system prompt and ask the LLM to answer
based only on the provided context. This is the standard RAG generation pattern.

In [8]:
from ollama import chat

SYSTEM_PROMPT = """\
You are a precise HR analyst. Answer the user's question using ONLY the context \
provided below. If the answer is not in the context, say "Not found in the document.". \
Be concise and quote exact numbers when available.

CONTEXT:
{context}"""


def rag_answer(question: str, k: int = TOP_K) -> dict:
    """Retrieve relevant chunks and generate an answer with the LLM."""
    retrieved = retrieve(question, k=k)
    context = "\n\n---\n\n".join(chunk for chunk, _ in retrieved)

    response = chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
            {"role": "user", "content": question},
        ],
        options={"temperature": 0},
    )
    return {
        "question": question,
        "answer": response.message.content.strip(),
        "chunks_used": len(retrieved),
    }


# Single question walkthrough
demo = rag_answer("What is the Q4 2024 attrition rate at Contoso Corp?")
print(f"Q: {demo['question']}")
print(f"A: {demo['answer']}")

Q: What is the Q4 2024 attrition rate at Contoso Corp?
A: The Q4 2024 attrition rate at Contoso Corp is 4.2%.


## 6. Test questions — full pipeline evaluation

Five questions with answers directly verifiable from `contoso_report_q4_2024.md`.
Each answer is checked against ground truth to validate end-to-end correctness.

In [9]:
# Questions and the key fact that must appear in the answer
TEST_QUESTIONS = [
    (
        "What is the total headcount of Contoso Corp as of Q4 2024?",
        "214",
    ),
    (
        "What was the Q4 2024 attrition rate at Contoso?",
        "4.2",
    ),
    (
        "Which department has the highest number of employees?",
        "Engineering",
    ),
    (
        "How many new hires are planned for Q1 2025?",
        "24",
    ),
    (
        "What is the average base salary at Contoso Corp?",
        "81,300",
    ),
]

print(f"Running {len(TEST_QUESTIONS)} questions through the RAG pipeline…\n")

passed = 0
for question, expected_keyword in TEST_QUESTIONS:
    result = rag_answer(question)
    ok = expected_keyword.lower() in result["answer"].lower()
    passed += int(ok)
    mark = "✓" if ok else "✗"
    print(f"{mark} Q: {question}")
    print(f"   A: {result['answer']}")
    if not ok:
        print(f"   Expected keyword '{expected_keyword}' not found in answer.")
    print()

print(f"Result: {passed}/{len(TEST_QUESTIONS)} questions answered correctly")
print("Data residency: ✅ All processing stayed on this machine")

Running 5 questions through the RAG pipeline…

✓ Q: What is the total headcount of Contoso Corp as of Q4 2024?
   A: The total headcount of Contoso Corp as of Q4 2024 is 214 employees.

✓ Q: What was the Q4 2024 attrition rate at Contoso?
   A: The Q4 2024 attrition rate at Contoso Corp was 4.2%.

✓ Q: Which department has the highest number of employees?
   A: The department with the highest number of employees is Engineering, with 72 employees.

✓ Q: How many new hires are planned for Q1 2025?
   A: The document states that there is a target of +24 new hires by 31 March 2025.

✓ Q: What is the average base salary at Contoso Corp?
   A: The average base salary at Contoso Corp is £81,300.

Result: 5/5 questions answered correctly
Data residency: ✅ All processing stayed on this machine
